In [1]:
from transformers import DistilBertTokenizer, DistilBertModel
import torch
from sklearn.cluster import KMeans
import numpy as np
from docx import Document

def extractive_summary(text, num_clusters=5):
    tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
    model = DistilBertModel.from_pretrained('distilbert-base-uncased')

    sentences = text.split('. ')
    sentence_embeddings = []

    for sentence in sentences:
        inputs = tokenizer(sentence, return_tensors='pt', padding=True, truncation=True)
        outputs = model(**inputs)
        hidden_states = outputs.last_hidden_state.mean(dim=1).detach().numpy()
        sentence_embeddings.append(hidden_states)

    kmeans = KMeans(n_clusters=num_clusters)
    kmeans = kmeans.fit(np.concatenate(sentence_embeddings))
    avg = []
    for j in range(num_clusters):
        idx = np.where(kmeans.labels_ == j)[0]
        avg.append(np.mean(idx))
    closest = np.array(avg).argsort()[:num_clusters]
    summary = '. '.join([sentences[i] for i in closest])
    
    return summary

# Read the content from the Word document
try:
    document = Document("/Users/Jolam/Desktop/Text Chapter one.docx")
    full_text = []
    for paragraph in document.paragraphs:
        full_text.append(paragraph.text)

    text = '\n'.join(full_text)

    # Dynamically determine number of clusters
    num_clusters = min(len(text.split('. ')) // 10, 10)

    print("Summary:")
    print(extractive_summary(text, num_clusters))

except Exception as e:
    print(f"An error occurred: {e}")

Summary:
Here a Protestant attested his belief; there
a Leaguer cursed Henry IV.; elsewhere some bourgeois has carved the
insignia of his _noblesse de cloches_, symbols of his long-forgotten
magisterial glory. I

There are houses in certain provincial towns whose aspect inspires
melancholy, akin to that called forth by sombre cloisters, dreary
moorlands, or the desolation of ruins. Houses three centuries old are still
solid, though built of wood, and their divers aspects add to the
originality which commends this portion of Saumur to the attention of
artists and antiquaries.

It is difficult to pass these houses without admiring the enormous oaken
beams, their ends carved into fantastic figures, which crown with a
black bas-relief the lower floor of most of them. These
low rooms, which have no shop-frontage, no show-windows, in fact
no glass at all, are deep and dark and without interior or exterior
decoration. This street--now
little frequented, hot in summer, cold in winter, dark in 